In [1]:
import random
import sys
sys.path.append('../core')
from DirDatasetBuilder import *

/apps/external/apps/jupyter/3.1.1/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/apps/external/apps/jupyter/3.1.1/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
2026-02-15 22:08:39.297947: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-15 22:08:39.311953: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-15 22:08:39.316261: E external/local_xla/xla/stream_executor/cuda/

#### Rhea Data

In [2]:
# Reference the location where Rhea's data is
swissprot_db = "acquired_data/06182025_rhea2uniprot_sprot.tsv"
other_db = "acquired_data/06182025_rhea2xrefs.tsv"
rhea_ids_db = "acquired_data/06182025_rhea-directions.tsv"
rhea_smiles_db = "acquired_data/06182025_rhea-reaction-smiles.tsv"
rhea_transport_ids = "acquired_data/06182025_transport_rhea.tsv"
rhea_parent_ids = "acquired_data/06182025_rhea-relationships.tsv"

# Determine a name to save the final file
final_output_name = "02152026_directionality_dataset"

### Prepare main directionality dataset

#### Get cross-references

In [3]:
full_rhea_ids = pd.read_csv(rhea_ids_db, delimiter='\t')
print(f"Total Reactions in Rhea: {len(full_rhea_ids['RHEA_ID_LR'])+len(full_rhea_ids['RHEA_ID_RL'])}")

directionality_dataset_1 = get_crosslinks(swissprot_db, other_db) # as of now, the feasibility dataset is a dictionary
print(f"  Total Entries with Crosslinks: {len(directionality_dataset_1['RHEA_ID'])}")

Total Reactions in Rhea: 35566
  Total Entries with Crosslinks: 27310


#### Add reaction SMILES

In [4]:
directionality_dataset_2 = add_smiles(directionality_dataset_1, rhea_smiles_db)  # it is now a dataframe
print(f"  Total Entries with Valid SMILES: {len(directionality_dataset_2)}")

  Total Entries with Valid SMILES: 26281


#### Remove transport reactions

In [5]:
print(f"  Checking Transport reactions:")
directionality_dataset_3 = remove_transport_reactions(directionality_dataset_2, rhea_transport_ids)
print(f"  Total Entries without Transport Reactions: {len(directionality_dataset_3)}")

  Checking Transport reactions:
   -Total number of UN Transport IDs: 1522
  Total Entries without Transport Reactions: 24911


#### Pre-process entries

In [6]:
print(f"  Complementing the dataset:")
directionality_dataset_4 = complement_entries(directionality_dataset_3)

  Complementing the dataset:
   -Example: 
      RHEA_ID direction feasible?  \
5958    10001        LR       Yes   

                                             rxn_smiles  \
5958  CCCCC(N)=O.[H]O[H]>>CCCCC(=O)[O-].[H][N+]([H])...   

                          can_rxn_smiles          rxn_products  
5958  CCCCC(N)=O.O>>CCCCC(=O)[O-].[NH4+]  CCCCC(=O)[O-].[NH4+]  


#### Remove entries with issues in template extraction

In [7]:
print(f"  Checking reactions that can undergo template extraction:")
directionality_dataset_5, template_dataset, exceptions = template_checker(directionality_dataset_4)

  Checking reactions that can undergo template extraction:


Processing reactions: 100%|██████████| 24911/24911 [04:02<00:00, 102.70it/s]

    -Successfully processed: 24387 reactions
    -Exceptions: 524 reactions


In [8]:
# Take a look at issues identified
random.choice(exceptions)

{'rhea_id': 30973,
 'error': 'temp as None',
 'smiles': 'CC(C)=CCC/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C=C(C)/C=C/C=C(C)/C=C/C=C(\\C)CCC=C(C)C>>CC(C)=CCC/C(C)=C/C=C\\C(C)=C/C=C/C(C)=C/C=C/C=C(C)/C=C/C=C(C)\\C=C/C=C(\\C)CCC=C(C)C'}

#### Reaction Chemistry Checker Validation

In [9]:
print(f"Running Reaction Chemistry Checker validation")
directionality_dataset_6, failed_validation_entries = reaction_checker_validation(directionality_dataset_5)

Running Reaction Chemistry Checker validation
Using 50 processes for parallel processing
Processing reactions in parallel...


Processing reactions: 100%|██████████| 24387/24387 [01:34<00:00, 257.61it/s]


    -Processed 24387 reactions. Accuracy: 99.07%
    -Total remaining data: 24161
RDChiral count = 22166
Achiral count = 1802
RDKit count = 419


In [11]:
# Check failed reactions
random_row = failed_validation_entries[['RHEA_ID', 'direction', 'feasible?', 'can_rxn_smiles', 'debug_info']].sample(n=1)
for ix, row in random_row.iterrows():
    print(f"RHEA_ID: {row['RHEA_ID']}")
    print(f"Feasible?: {row['feasible?']}")
    print(f"Direction: {row['direction']}")
    print(f"Can_rxn_smiles: {row['can_rxn_smiles']}")
    print(f"Debug info: {row['debug_info']}")

RHEA_ID: 38409
Feasible?: No
Direction: RL
Can_rxn_smiles: CCCCC/C=C\C/C=C\CCCCCCCC(=O)OC[C@@H](CO)OC(=O)CCCCCCC/C=C\C/C=C\CCCCC.CCCCC/C=C\C/C=C\CCCCCCCC(=O)[O-]>>CCCCC/C=C\C/C=C\CCCCCCCC(=O)OCC(COC(=O)CCCCCCC/C=C\C/C=C\CCCCC)OC(=O)CCCCCCC/C=C\C/C=C\CCCCC.O
Debug info: Match not found



#### Save Directionality Dataset

In [12]:
# Save as normal csv
directionality_dataset_6.to_csv(f"../data/{final_output_name}.csv", index=False)
failed_validation_entries.to_csv(f"validation_output/{final_output_name}_ReactChemChecker_failed_validation_entries.csv", index=False)
print(f"Directionality Dataset saved!")

# Save as pkl file with fingerprint
precompute_fingerprints(final_output_name, directionality_dataset_6)

Directionality Dataset saved!
Done! Precomputed fingerprints stored in ../data/02152026_directionality_dataset.pkl


### Prepare validation dataset

#### Remove parent reactions

In [13]:
directionality_dataset_7 = remove_parent_ids(directionality_dataset_6, rhea_ids_db, rhea_parent_ids)

   -Number of (Non-Unique) Parent IDs: 19584
   -Number of Unique Parent IDs: 5564
   -Total UN IDs in the Feasibility Dataset: 12249
   -Number of parent IDs in the Feasibility Dataset: 830 - [65612, 65632, 65692, 32939, 32943]
   -Original length: 24161
   -Final length: 22534


#### Remove unique chemistries

In [14]:
directionality_dataset_8 = remove_unique_temp(directionality_dataset_7, template_dataset)

   -Non-unique templates (all kept):16080


#### Save validation dataset

In [15]:
# Save as normal csv
directionality_dataset_8.to_csv(f"../data/CV_{final_output_name}.csv", index=False)

# Save as pkl file with fingerprint
precompute_fingerprints(f'CV_{final_output_name}', directionality_dataset_8)

Done! Precomputed fingerprints stored in ../data/CV_02152026_directionality_dataset.pkl
